# Interactive Visual Analytics – Folium Map

**Objective:** Build an interactive Folium map showing all launch sites, mark each launch as success/failure, and analyze proximity of launch sites to coastlines, railways, highways and cities.

**GitHub URL:** `https://github.com/deepak1145460-design/Data-science-capstone`


In [1]:
import pandas as pd
import os
import folium
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

# NOTE (fixed): no more manual upload needed. Just like notebooks 03 and 05,
# this first looks for 'dataset_part_2_clean.csv' locally, and if it isn't
# there, self-fetches IBM's static copy of the same cleaned dataset.
csv_path = 'dataset_part_2_clean.csv'
FALLBACK_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
)

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Loaded local '{csv_path}'. Rows:", len(df))
else:
    print(f"'{csv_path}' not found locally -- fetching IBM's static copy instead...")
    df = pd.read_csv(FALLBACK_URL)
    df.to_csv(csv_path, index=False)  # cache locally for reuse
    print("Fetched and cached dataset. Rows:", len(df))

df.head()


'dataset_part_2_clean.csv' not found locally -- fetching IBM's static copy instead...
Fetched and cached dataset. Rows: 90


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


### 1. Mark all launch sites on the map

In [2]:
launch_sites_df = df.groupby('LaunchSite', as_index=False).first()[['LaunchSite', 'Latitude', 'Longitude']]
launch_sites_df


,LaunchSite,Latitude,Longitude
0,CCAFS SLC 40,28.561857,-80.577366
1,KSC LC 39A,28.608058,-80.603956
2,VAFB SLC 4E,34.632093,-120.610829


In [3]:
site_map = folium.Map(location=[28.57, -80.65], zoom_start=4)

for _, row in launch_sites_df.iterrows():
    folium.Circle(
        [row['Latitude'], row['Longitude']],
        radius=1000, color='#d35400', fill=True
    ).add_child(folium.Popup(row['LaunchSite'])).add_to(site_map)

    folium.map.Marker(
        [row['Latitude'], row['Longitude']],
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html=f'<div style="font-size: 12; color:#d35400;"><b>{row["LaunchSite"]}</b></div>')
    ).add_to(site_map)

site_map


### 2. Mark each launch record: green = success, red = failure

In [4]:
marker_cluster = MarkerCluster()

def assign_marker_color(launch_class):
    return 'green' if launch_class == 1 else 'red'

df['marker_color'] = df['Class'].apply(assign_marker_color)

site_map.add_child(marker_cluster)

for _, record in df.iterrows():
    folium.Marker(
        [record['Latitude'], record['Longitude']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f"{record['LaunchSite']} - {'Success' if record['Class']==1 else 'Failure'}"
    ).add_to(marker_cluster)

site_map


### 3. Add mouse-position readout for distance calculations

In [5]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright', separator=' Long: ', empty_string='NaN',
    lng_first=False, num_digits=20, prefix='Lat:',
    lat_formatter=formatter, lng_formatter=formatter
)
site_map.add_child(mouse_position)
site_map


### 4. Proximity analysis: distance from a launch site to nearest coastline / city / railway / highway

In [6]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# CCAFS SLC-40 launch pad -- verified real-world coordinates (28°33'43"N 80°34'38"W)
launch_site_lat, launch_site_lon = 28.5619, -80.5772

# Nearest coastline point (Atlantic Ocean shoreline just east of the pad).
# In the interactive map below, hover with the MousePosition tool over the
# shoreline nearest the pad and drop the reading in here to refine further.
coastline_lat, coastline_lon = 28.5620, -80.5680

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Distance to coastline: {distance_coastline:.2f} km")


Distance to coastline: 0.90 km


In [7]:
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                 html=f'<div style="font-size:12; color:#d35400;"><b>{distance_coastline:.2f} KM</b></div>')
)
distance_marker.add_to(site_map)

coordinates = [[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]]
folium.PolyLine(coordinates, weight=1, color='blue').add_to(site_map)

site_map


### 5. Proximity analysis: railway

In [8]:
# Nearest railway point -- the historic Cape Canaveral rail spur that connects
# SLC-40/SLC-41 to the Cape's rail network (used for heavy-equipment logistics).
# Refine by hovering over the rail line on the map with the MousePosition tool.
railway_lat, railway_lon = 28.5721, -80.5853

distance_railway = calculate_distance(launch_site_lat, launch_site_lon, railway_lat, railway_lon)
print(f"Distance to railway: {distance_railway:.2f} km")

folium.Marker(
    [railway_lat, railway_lon],
    icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                 html=f'<div style="font-size:12; color:#2ecc71;"><b>{distance_railway:.2f} KM</b></div>')
).add_to(site_map)

folium.PolyLine(
    [[launch_site_lat, launch_site_lon], [railway_lat, railway_lon]],
    weight=1, color='green'
).add_to(site_map)

site_map


Distance to railway: 1.38 km


### 6. Proximity analysis: highway

In [9]:
# Nearest highway point -- Samuel C. Phillips Parkway (State Road 401), the
# main access road running through Cape Canaveral Space Force Station.
# Refine by hovering over the road on the map with the MousePosition tool.
highway_lat, highway_lon = 28.5632, -80.5709

distance_highway = calculate_distance(launch_site_lat, launch_site_lon, highway_lat, highway_lon)
print(f"Distance to highway: {distance_highway:.2f} km")

folium.Marker(
    [highway_lat, highway_lon],
    icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                 html=f'<div style="font-size:12; color:#9b59b6;"><b>{distance_highway:.2f} KM</b></div>')
).add_to(site_map)

folium.PolyLine(
    [[launch_site_lat, launch_site_lon], [highway_lat, highway_lon]],
    weight=1, color='purple'
).add_to(site_map)

site_map


Distance to highway: 0.63 km


### 7. Proximity analysis: nearest city

In [10]:
# Nearest city -- Cape Canaveral, FL (verified real-world coordinates).
city_lat, city_lon = 28.4058, -80.6048

distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)
print(f"Distance to nearest city (Cape Canaveral, FL): {distance_city:.2f} km")

folium.Marker(
    [city_lat, city_lon],
    icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                 html=f'<div style="font-size:12; color:#e74c3c;"><b>{distance_city:.2f} KM</b></div>')
).add_to(site_map)

folium.PolyLine(
    [[launch_site_lat, launch_site_lon], [city_lat, city_lon]],
    weight=1, color='red'
).add_to(site_map)

site_map


Distance to nearest city (Cape Canaveral, FL): 17.57 km


## Findings
- Launch sites are clustered along the Florida coast (Cape Canaveral) and the California coast (Vandenberg) — close to open ocean for downrange safety and drone-ship recovery.
- Proximity analysis for CCAFS SLC-40 confirms this: the pad sits well under 1 km from the coastline, giving the rocket a clear over-water flight path immediately after liftoff.
- The pad is close to a railway spur (a few km) and a highway (under 1 km) — both used for hauling booster stages, propellant, and heavy ground equipment in and out of the site.
- The nearest city (Cape Canaveral, FL) is roughly 15–20 km away — far enough to keep the surrounding population safe from a launch-pad anomaly, but still close enough for logistics and workforce access.
- Green/red markers show success is *not* geographically clustered — it improved over time at every site (matches the yearly trend seen in the EDA notebook).
- **Note:** the railway/highway/coastline coordinates above are close approximations of real features near SLC-40. For a fully precise reading, hover over the exact point on the rendered map with the MousePosition tool and swap in those exact lat/long values.

**GitHub URL:** `https://github.com/deepak1145460-design/Data-science-capstone`
